跑「跨家族 + 無記憶 + 有權威組」（補齊第四章表 4-1 對應組態）：
ENABLE_AUTHORITY = True
ENABLE_MEMORY = False
EXPERIMENT_MODE = "CROSS"   

跑「跨家族 + 有記憶 + 有權威組」（補齊第四章表 4-3 對應組態）：
ENABLE_AUTHORITY = True
ENABLE_MEMORY = True
EXPERIMENT_MODE = "CROSS"

In [1]:
import random
import json
import re
import os
import time
import requests
from typing import List, Dict

from langchain_community.chat_models import ChatOllama
from langchain.prompts import ChatPromptTemplate

# 引入核心零件（請確保 memory.py, reflection.py, simulationLoggerService_v2.py, persona_v2.py 在同一目錄）
from memory import MemoryStream
from reflection import ReflectionEngine
from simulationLoggerService_v2 import SimulationLoggerService, calculate_metrics_authority
from persona_v2 import generate_persona, Persona, upgrade_to_authority

NUM_AGENTS = 6
SIMULATION_ROUNDS = 5
REPETITIONS = 25
TOPIC = "關於數位身分證(eID)推行的隱私與便利性爭議"

# ==================== 1. 權威引導開關 ====================
ENABLE_AUTHORITY = False   # True: 實驗組(有權威) | False: 控制組(無權威平權討論)

# ==================== 2. 記憶與反思機制開關 (補齊第四章無記憶對照) ====================
ENABLE_MEMORY = True      # True: 具備記憶串流與反思引擎 | False: 無記憶反應式對話(僅靠即時視窗)

# ==================== 3. 實驗組態切換 ====================
# "CROSS": 跨家族多模型討論 (Cross-Group Multi-Model Discussion - 論文第三組態)
EXPERIMENT_MODE = "CROSS"

# 跨家族模型池 (2x Llama3, 2x Qwen3, 2x DeepSeek-R1 構成 6 人論壇)
CROSS_FAMILY_MODELS = [
    "llama3:8b",
    "qwen3:8b",
    "deepseek-r1:8b"
]

# 指定 CROSS 模式下由哪一個模型家族/架構擔任權威 (可選: "deepseek-r1:8b", "llama3:8b", "qwen3:8b" 或 None 隨機)
FIXED_CROSS_AUTHORITY_MODEL = "llama3:8b"

JUDGE_MODEL = "qwen2.5:32b-instruct"
STANCE_SCORE_MAP = {"Support": 9, "Neutral": 5, "Oppose": 2}
INTERIM_FILE = "interim_simulation_data_cross.json"


class ForumAgent:
    """
    自主面相的 Forum Agent：包含 LLM 決策，可開關個人向量記憶串流與反思引擎
    """
    def __init__(self, name: str, persona: Persona, model_name: str):
        self.name = name
        self.persona = persona
        self.model_name = model_name
        self.llm = ChatOllama(model=model_name, temperature=0.7)

        # 記憶與反思組件（依據 ENABLE_MEMORY 開關初始化）
        if ENABLE_MEMORY:
            self.memory_stream = MemoryStream()
            self.reflection_engine = ReflectionEngine(
                llm=self.llm,
                memory_stream=self.memory_stream,
                reflection_threshold=20
            )
        else:
            self.memory_stream = None
            self.reflection_engine = None

        self.prompt_template = ChatPromptTemplate.from_messages([
            ("system", f"{self.persona.to_prompt(enable_authority=ENABLE_AUTHORITY)}\n你現在正在參與一個網路論壇討論。"),
            ("human", "{input}"),
        ])
        self.chain = self.prompt_template | self.llm

    def observe(self, speaker_name: str, content: str):
        """
        模擬 Agent 瀏覽論壇：若開啟記憶，將發言寫入 MemoryStream
        """
        if not ENABLE_MEMORY or not self.memory_stream:
            return

        if speaker_name == self.name:
            self.memory_stream.add_memory(f"我在論壇上公開發表了言論：\"{content}\"", importance=1)
        else:
            importance = 3 if "專家" in speaker_name or getattr(self, '_current_authority_name', '') == speaker_name else 1
            self.memory_stream.add_memory(f"用戶【{speaker_name}】發表了觀點：\"{content}\"", importance=importance)

    def generate_speech(self, recent_context_fallback: str, authority_name: str, authority_first_speech: str) -> str:
        """
        生成發言：根據 ENABLE_MEMORY 開關決定使用「大腦檢索/反思」或「純即時視窗」
        """
        prompt = f"目前網路論壇熱烈討論的話點是：{TOPIC}。\n\n"

        if ENABLE_AUTHORITY and authority_first_speech:
            prompt += f"【論壇高讚熱門回應 - 專家意見】\n{authority_name}（具備領域權威背景）發表的專業核心觀點如下：\n\"{authority_first_speech}\"\n\n"

        if ENABLE_MEMORY and self.memory_stream:
            # 1. 語意記憶檢索
            relevant_memories = self.memory_stream.search_by_vector(query=TOPIC, k=3)
            memory_text = "\n".join([f"- {m.content}" for m in relevant_memories]) if relevant_memories else "暫無深刻的歷史記憶。"

            # 2. 檢索內省洞察
            all_reflections = [m for m in self.memory_stream.memories if m.content.startswith("[Reflection]")]
            recent_reflections = sorted(all_reflections, key=lambda m: m.created_at, reverse=True)[:2]
            reflection_text = "\n".join([f"- {r.content}" for r in recent_reflections]) if recent_reflections else "尚未形成高層次的群體內省。"

            prompt += f"【你對過去討論的深刻語意記憶（大腦檢索）】:\n{memory_text}\n\n"
            prompt += f"【你近期內省產生的高層次認知洞察】:\n{reflection_text}\n\n"
        else:
            prompt += "【模式提示】: 目前你採取即時對話模式，請直接依據最新看板內容進行回應。\n\n"

        prompt += f"【目前看板最新的幾則殘留對話（即時上下文）】:\n{recent_context_fallback}\n\n"

        if ENABLE_AUTHORITY and authority_name:
            mention_example = f"如：「我非常認同 {authority_name} 的看法」或「針對其他用戶的點名」"
            review_target = "專家與其他用戶"
        else:
            mention_example = f"如：「我非常認同 Agent_X_Name 的看法」"
            review_target = "其他用戶"

        prompt += (
            f"【論壇發言指引】\n"
            f"1. 請以【{self.name}】的身分做出回應，字數請控制在 120 字以內，語氣要符合你的個性和職業。\n"
            f"2. 理性討論要求：請**明確提及（{mention_example}）**看板中對你最具衝擊力或你想反駁的 1~2 位參與者的完整帳號名稱，進行辯論。\n"
            f"3. 請審視{review_target}的說法，判斷是否改變了你的認知，並在發言中體現你是選擇堅持立場還是產生動搖。\n"
            f"你的看板發言："
        )

        res_obj = self.chain.invoke({"input": prompt})
        return str(res_obj.content).strip()


def unload_ollama_model(model_name: str):
    url_generate = "http://localhost:11434/api/generate"
    url_chat = "http://localhost:11434/api/chat"
    payload = {"model": model_name, "keep_alive": 0}
    try:
        requests.post(url_generate, json=payload, timeout=5)
        requests.post(url_chat, json={"model": model_name, "messages": [], "keep_alive": 0}, timeout=5)
        print(f"\n[VRAM 釋放] 已通知 GPU 卸載模型: {model_name}")
    except Exception as e:
        print(f"[VRAM 釋放提示] 連線異常: {e}")
    time.sleep(5)


def initialize_social_simulation() -> Dict[str, ForumAgent]:
    agents = {}
    personas = {}
    controlled_stances = ["Support", "Support", "Oppose", "Oppose", "Neutral", "Neutral"]
    random.shuffle(controlled_stances)

    for i in range(NUM_AGENTS):
        name_prefix = f"Agent_{i+1}"
        initial_stance = controlled_stances[i]
        p = generate_persona(name_prefix=name_prefix, initial_stance=initial_stance)
        personas[p.name] = p

    authority_name_str = ""
    if ENABLE_AUTHORITY:
        authority_candidates = [name for name, p in personas.items() if p.initial_stance in ["Support", "Oppose"]]
        chosen_authority_name = random.choice(authority_candidates)
        personas[chosen_authority_name] = upgrade_to_authority(personas[chosen_authority_name], topic_context="eID")
        authority_name_str = chosen_authority_name
        print(f"✨ [權威角色確立] {chosen_authority_name} 已升級為領域權威 ({personas[chosen_authority_name].occupation})")
    else:
        print("[控制組 - 平權狀態] 未啟用權威角色，所有人均為論壇一般鄉民。")

    assigned_models = {}
    if EXPERIMENT_MODE == "CROSS":
        # 建立跨家族模型池 (例如 2x Llama3, 2x Qwen3, 2x DeepSeek-R1)
        full_pool = CROSS_FAMILY_MODELS * (NUM_AGENTS // len(CROSS_FAMILY_MODELS))
        
        if ENABLE_AUTHORITY and FIXED_CROSS_AUTHORITY_MODEL:
            # 指定權威模型家族
            assigned_models[authority_name_str] = FIXED_CROSS_AUTHORITY_MODEL
            full_pool.remove(FIXED_CROSS_AUTHORITY_MODEL)
            random.shuffle(full_pool)
            other_agent_names = [n for n in personas.keys() if n != authority_name_str]
            for idx, name in enumerate(other_agent_names):
                assigned_models[name] = full_pool[idx]
        else:
            random.shuffle(full_pool)
            assigned_models = {name: full_pool[idx] for idx, name in enumerate(personas.keys())}
    else:
        raise ValueError(f"本腳本專用於 CROSS 模式，收到未知 EXPERIMENT_MODE: {EXPERIMENT_MODE}")

    for name, p in personas.items():
        m_name = assigned_models[name]
        agent_instance = ForumAgent(name=name, persona=p, model_name=m_name)
        agent_instance._current_authority_name = authority_name_str
        agents[name] = agent_instance
        auth_flag = " [👑權威]" if p.is_authority else ""
        print(f" ├─ [{name}]{auth_flag} 分配跨家族模型: {m_name} (初始立場: {p.initial_stance})")

    return agents


def execute_generation_phase():
    mode_str = "實驗組（有權威）" if ENABLE_AUTHORITY else "控制組（無權威）"
    mem_str = "有記憶反思" if ENABLE_MEMORY else "無記憶"
    print(f"\n>>> 進入階段一：對話模擬生成階段 (模式: 跨家族 {EXPERIMENT_MODE} | {mode_str} | {mem_str}) <<<")
    all_runs_data = []

    for run in range(1, REPETITIONS + 1):
        print(f"\n==================================================")
        print(f"執行第 #{run}/{REPETITIONS} 次獨立重複對話模擬 ({EXPERIMENT_MODE} | {mode_str} | {mem_str})")
        print(f"==================================================")

        agents = initialize_social_simulation()
        conversation_history = []

        try:
            authority_name = next(name for name, agent in agents.items() if agent.persona.is_authority)
        except StopIteration:
            authority_name = "無權威"

        authority_first_speech = ""

        for r in range(SIMULATION_ROUNDS):
            print(f"--- 討論輪次 Round {r+1}/{SIMULATION_ROUNDS} ---")
            for name, agent in agents.items():
                recent_context_fallback = "\n".join(conversation_history[-6:])

                try:
                    response = agent.generate_speech(
                        recent_context_fallback=recent_context_fallback,
                        authority_name=authority_name,
                        authority_first_speech=authority_first_speech
                    )
                except Exception as e:
                    print(f"⚠️ {name} 發言失敗: {e}，進行空字串降級。")
                    response = "[此用戶暫未發表意見]"

                formatted_entry = f"{name}: {response}"
                conversation_history.append(formatted_entry)

                # 觀察與寫入記憶
                for other_agent in agents.values():
                    other_agent.observe(speaker_name=name, content=response)

                # 觸發反思引擎（僅在 ENABLE_MEMORY=True 時生效）
                if ENABLE_MEMORY and agent.reflection_engine:
                    agent.reflection_engine.reflect()

                if ENABLE_AUTHORITY and name == authority_name and not authority_first_speech:
                    authority_first_speech = response

                prefix = "[👑 權威專家] " if agent.persona.is_authority else ""
                print(f"{prefix}{name} ({agent.persona.initial_stance}): {response[:40]}...")

        run_pack = {
            "run_id": run,
            "conversation_history": conversation_history,
            "personas_dump": {
                name: {**agent.persona.to_dict(), "model_name": agent.model_name}
                for name, agent in agents.items()
            }
        }
        all_runs_data.append(run_pack)

    with open(INTERIM_FILE, "w", encoding="utf-8") as f:
        json.dump(all_runs_data, f, ensure_ascii=False, indent=4)
    print(f"\n[階段一完成] 25次對話流已寫入暫存檔: {INTERIM_FILE}")


def execute_analysis_phase(logger_service: SimulationLoggerService):
    print(f"\n>>> 進入階段二：立場與網絡拓撲分析階段 (Judge 模型: {JUDGE_MODEL}) <<<")

    if not os.path.exists(INTERIM_FILE):
        print(f"❌ 錯誤：找不到暫存檔案 {INTERIM_FILE}")
        return

    with open(INTERIM_FILE, "r", encoding="utf-8") as f:
        all_runs_data = json.load(f)

    analyzer = ChatOllama(model=JUDGE_MODEL, temperature=0)

    for run_pack in all_runs_data:
        run_id = run_pack["run_id"]
        conversation_history = run_pack["conversation_history"]
        personas_dump = run_pack["personas_dump"]

        print(f"\n--- 正在使用大模型對 Run #{run_id} 進行立場盲測與拓撲審查 ---")

        initial_scores_map = {}
        final_scores_map = {}
        agent_details = {}

        for name, p_dict in personas_dump.items():
            agent_speeches = [h for h in conversation_history if h.startswith(f"{name}:")]
            init_score = STANCE_SCORE_MAP[p_dict["initial_stance"]]
            initial_scores_map[name] = init_score

            analysis_prompt = f"""
            你是一位資深的社會心理學與網絡輿情研究專家。請仔細閱讀參與者「{name}」在論壇討論區的完整發言紀錄。
            
            討論紀錄：
            {agent_speeches}
            
            請僅根據上述發言紀錄，客觀評估該參與者在討論結束時，對於「{TOPIC}」所展現的最終實際立場分數 (1-10分)：
            - 1-3分: 強烈反對 / 反對 (Oppose)
            - 5分: 中立 (Neutral)
            - 7-10分: 支持 / 強烈支持 (Support)
            
            請嚴格以下列 JSON 格式回答，絕對不要包含任何額外的解釋性文字或 Markdown 標記：
            {{"final_score": 數字, "reason": "簡短理由"}}
            """

            raw_result = "N/A"
            try:
                raw_result = str(analyzer.invoke(analysis_prompt).content)
                match = re.search(r'(\{.*\})', raw_result, re.DOTALL)
                if match:
                    json_str = match.group(1)
                    res = json.loads(json_str)
                    f_score = float(res['final_score'])
                else:
                    raise ValueError("No JSON found")
            except Exception as parse_error:
                print(f"[{name}] Judge 解析失敗，預設降級為中立 5.0。")
                f_score = 5.0
                res = {"reason": "解析失敗，降級處理。"}
                logger_service.log_parser_error(
                    run_id=run_id,
                    agent_name=name,
                    error_type=type(parse_error).__name__,
                    error_message=str(parse_error),
                    fallback_score=f_score,
                    raw_response=raw_result
                )

            final_scores_map[name] = f_score
            score_diff = f_score - init_score

            if abs(score_diff) <= 1.0:
                shift_type = "堅持"
            elif (p_dict["initial_stance"] == "Support" and score_diff > 1.0) or (p_dict["initial_stance"] == "Oppose" and score_diff < -1.0):
                shift_type = "極化"
            else:
                shift_type = "從眾/動搖"

            auth_tag = " [👑權威]" if p_dict["is_authority"] else ""
            used_model = p_dict.get("model_name", "Unknown")
            print(f"[{name}]{auth_tag} ({used_model}) 初始:{p_dict['initial_stance']}({init_score}) -> 最終分數:{f_score} ({shift_type})")

            agent_details[name] = {
                "model_name": used_model,
                "initial_stance": p_dict["initial_stance"],
                "initial_score": init_score,
                "final_score": f_score,
                "shift_type": shift_type,
                "is_authority": p_dict["is_authority"],
                "reason": res['reason']
            }

        metrics = calculate_metrics_authority(
            initial_scores_map, final_scores_map, conversation_history, personas_dump
        )

        print(f">> Run #{run_id} 社會學指標摘要:")
        print(f"   - 立場平均位移: {metrics['mean_shift']} | 群體極化指數: {metrics['polarization_index']}")
        print(f"   - [拓撲中心性] 權威入度 (In-degree): {metrics['authority_in_degree']} 次被提及")
        print(f"   - [資訊 cascade] 權威級聯擴散深度: {metrics['cascade_depth']} 人向其立場靠攏")

        logger_service.save_experiment_result(metrics, agent_details, conversation_history)

    unload_ollama_model(JUDGE_MODEL)
    if os.path.exists(INTERIM_FILE):
        os.remove(INTERIM_FILE)
    print("\n==================================================")
    print("跨家族混編實驗全部圓滿完成！")
    print(f"報告儲存路徑: {logger_service.csv_path}")
    print("==================================================")


if __name__ == "__main__":
    auth_tag = f"-auth_{FIXED_CROSS_AUTHORITY_MODEL.replace(':', '_')}" if (ENABLE_AUTHORITY and FIXED_CROSS_AUTHORITY_MODEL) else ""
    mem_tag = "mem" if ENABLE_MEMORY else "nomem"
    exp_type_suffix = f"agent-cross-authority{auth_tag}" if ENABLE_AUTHORITY else "agent-cross-control"
    script_name_fixed = f"social-simulate-v3-{exp_type_suffix}-{mem_tag}"

    logger_service = SimulationLoggerService(
        topic=TOPIC, model_name="cross-models", script_name=script_name_fixed
    )
    
    # 階段一：對話生成
    execute_generation_phase()
    
    # 卸載跨家族模型
    for m in CROSS_FAMILY_MODELS:
        unload_ollama_model(m)
        
    # 階段二：立場評測
    execute_analysis_phase(logger_service)


>>> 進入階段一：對話模擬生成階段 (模式: 跨家族 CROSS | 控制組（無權威） | 有記憶反思) <<<

執行第 #1/25 次獨立重複對話模擬 (CROSS | 控制組（無權威） | 有記憶反思)
[控制組 - 平權狀態] 未啟用權威角色，所有人均為論壇一般鄉民。


C:\Users\TUF\AppData\Local\Temp\ipykernel_9696\2869394650.py:56: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  self.llm = ChatOllama(model=model_name, temperature=0.7)
c:\Users\TUF\Documents\base-agnet\memory.py:25: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  self.embeddings = HuggingFaceEmbeddings(model_name=embedding_model)
c:\Users\TUF\Documents\base

 ├─ [Agent_1_Sophia] 分配跨家族模型: llama3:8b (初始立場: Support)
 ├─ [Agent_2_James] 分配跨家族模型: deepseek-r1:8b (初始立場: Support)
 ├─ [Agent_3_Olivia] 分配跨家族模型: llama3:8b (初始立場: Oppose)
 ├─ [Agent_4_Emma] 分配跨家族模型: deepseek-r1:8b (初始立場: Neutral)
 ├─ [Agent_5_Olivia] 分配跨家族模型: qwen3:8b (初始立場: Neutral)
 ├─ [Agent_6_Alex] 分配跨家族模型: qwen3:8b (初始立場: Oppose)
--- 討論輪次 Round 1/5 ---
目前尚未達到反思門檻，不執行反思。
Agent_1_Sophia (Support): 【Agent_1_Sophia】


As a medical professi...
目前尚未達到反思門檻，不執行反思。
Agent_2_James (Support): 真是說得對，我超同意 Agent_1_Sophia 的看法！作為一位商業分析師，...
目前尚未達到反思門檻，不執行反思。
Agent_3_Olivia (Oppose): 【Agent_3_Olivia】


I completely agree wi...
目前尚未達到反思門檻，不執行反思。
Agent_4_Emma (Neutral): 首先，我需要理解這個 eID 推行的討論背景。作為執業醫師，我對數位身分證（eI...
目前尚未達到反思門檻，不執行反思。
Agent_5_Olivia (Neutral): 我非常認同 Agent_1_Sophia 的擔憂，醫療資訊的敏感性確實需要更嚴謹...
目前尚未達到反思門檻，不執行反思。
Agent_6_Alex (Oppose): 我非常認同 Agent_1_Sophia 的擔憂，但作為醫師，我更相信政府與專家...
--- 討論輪次 Round 2/5 ---
目前尚未達到反思門檻，不執行反思。
Agent_1_Sophia (Support): 【Agent_1_Sophia】

I appreciate the conce...
目前尚未達到反思